In [1]:
# Task 3
# Imports
import random
import numpy as np

In [2]:
# 2 - Updated functions
def initialization_population_constrained(pop_size, num_items=1001, k=10):
    population = []
    for x in range(pop_size):
        individual = [1] * k + [0] * (num_items - k)
        random.shuffle(individual)
        population.append(individual)
    return population

def mutation(individuals: list[list[int]], mutation_prob: float) -> list[list[int]]:
    """Performs bitflip mutation for each bit / gene in each individual with mutation probability of P_M."""
    mutated = []
    for individual in individuals:
        offspring = individual.copy()
        if np.random.uniform(0, 1) <= mutation_prob:
            ones = [i for i, bit in enumerate(individual) if bit == 1]
            zeros = [i for i, bit in enumerate(individual) if bit == 0]
            if ones and zeros:
                one_choice = random.choice(ones)
                zero_choice = random.choice(zeros)
                offspring[one_choice] = 0
                offspring[zero_choice] = 1
        mutated.append(offspring)
    return mutated

def crossover(parents: list[list[int]], crossover_prob: float, k: int) -> list[list[int]]:
    """Performs crossover and generates for parents 1 & 2, parents 3 & 4 etc. offspring. """
    offsprings = []
    for i in range(0, len(parents)-1, 2):
        parent1, parent2 = parents[i], parents[i+1]

        alpha = np.random.uniform(0, 1)
        if alpha <= crossover_prob:
            agree_ones = [] # both are 1
            disagree = [] # they differ

            for j in range(len(parent1)):
                if parent1[j] == parent2[j] == 1:
                    agree_ones.append(j)
                elif parent1[j] != parent2[j]:
                    disagree.append(j)

            offspring1 = [0] * len(parent1)
            offspring2 = [0] * len(parent2)
            for j in agree_ones:
                offspring1[j] = 1
                offspring2[j] = 1

            needed = k - len(agree_ones)
            chosen1 = random.sample(disagree, needed)
            chosen2 = random.sample(disagree, needed)

            for j in chosen1:
                offspring1[j] = 1

            for j in chosen2:
                offspring2[j] = 1
        else:
            offspring1 = parent1.copy()
            offspring2 = parent2.copy()

        offsprings.append(offspring1)
        offsprings.append(offspring2)

    if len(parents) % 2 == 1:
        offsprings.append(parents[-1].copy())

    return offsprings

In [4]:
# New SGA with updated functions
# 1
dataArr = np.genfromtxt("../data/data_1001.csv",
                    delimiter=",", dtype=int, skip_header=1)
display(dataArr)
item_values = dataArr[:, 0] # first column
item_weights = dataArr[:, 1] # second column

def fitness_knapsack(individuals):
    return [np.dot(individual, item_values) for individual in individuals]
# Parents
def parent_selection( individuals: list[list[int]], fitness_values: list[float] ) -> list[list[int]]:
    """Perform fitness-proportional parent selection such that more fit parents have a higher probability to be selected. """
    fitness_values = np.array(fitness_values, dtype=float)
    shifted = fitness_values - fitness_values.min() + 1
    fitness = np.array(shifted, dtype=float)
    probs = fitness / fitness.sum()
    idx = np.random.choice(len(individuals), size=len(individuals), replace=True, p=probs)
    return [individuals[i] for i in idx]


# Survivor selection
def survivor_selection( individuals: list[list[int]], offspring: list[list[int]],
                        fitness_individuals: list[float], fitness_offspring: list[float]) -> list[list[int]]:
    """Selects survivors based on the highest fitness values from the pooled population individuals and offspring. """
    combined = individuals + offspring
    combined_fitness = fitness_individuals + fitness_offspring
    fitness_ordered = np.argsort(combined_fitness)[::-1]

    return  [combined[i] for i in fitness_ordered[:len(individuals)]]

def display_generation_fitness( new_avg_fit: float, new_best_fit: float, iter: int ):    
    print(f"Generation {iter:_}")
    print("-" * 50)
    print(f"Avg. fit: {new_avg_fit:_.6f} | Best fit: {new_best_fit:_.6f}") 
    print("-" * 50 + "\n")

def simple_genetic_algorithm( params: dict[str, int | float], k:int ):
    """Complete version of the simple genetic algorithm (SGA).
    
    params:
        1. n_pop: Controls the population size.
        2. n_bits: Defines the number of bits in a single candidate solution.
        3. c_prob: Probability that crossover (recombination) occurs between parent chromosomes.
        4. m_prob: Determines how frequently random changes occur in the offspring.
        5. n_iter: Termination criteria. Fixed number of algorithm iterations or generations to produce.
    """
    avg_fitness_data, best_fitness_data = [], []

    population = initialization_population_constrained( params["n_pop"], params["n_bits"], k )
    curr_avg_fit, curr_best_fit = 0.0, 0.0
    best_individual = None

    for _ in range( params["n_iter"] ):
        fitness_values = fitness_knapsack(population)

        new_avg_fit, new_best_fit = np.mean(fitness_values), np.max(fitness_values)
        # display_generation_fitness( new_avg_fit, new_best_fit, i + 1 )
        
        avg_fitness_data.append( new_avg_fit )
        best_fitness_data.append( new_best_fit )
        

        if new_best_fit > curr_best_fit:
            curr_best_fit = new_best_fit
            best_individual = population[np.argmax(fitness_values)]

        if new_avg_fit > curr_avg_fit:
            curr_avg_fit = new_avg_fit
        
        parents = parent_selection(population, fitness_values)
        offspring = crossover( parents, params["c_prob"], k )
        offspring = mutation(offspring, params["m_prob"])
        survivors = survivor_selection(population, offspring, fitness_values, fitness_knapsack(offspring))    

        population = survivors

    return avg_fitness_data, best_fitness_data, best_individual

for k in [1001, 802, 400, 100, 10]:
    avgFitness, bestFitness, bestIndividual = simple_genetic_algorithm({ "n_pop": 30, "n_bits": 1001, "c_prob": 0.9, "m_prob": 0.1, "n_iter": 1000 }, k)
    print(f"K={k:>4} | Best fitness: {bestFitness[-1]:>10} | Avg value/item: {bestFitness[-1]/k:.2f}")

array([[  3300, 100000],
       [  6964,  10483],
       [  -681,   3407],
       ...,
       [  6619,  11030],
       [  3651,   7391],
       [ 12340,  18087]], shape=(1001, 2))

K=1001 | Best fitness:    5767936 | Avg value/item: 5762.17
K= 802 | Best fitness:    6121347 | Avg value/item: 7632.60
K= 400 | Best fitness:    4531200 | Avg value/item: 11328.00
K= 100 | Best fitness:    1444973 | Avg value/item: 14449.73
K=  10 | Best fitness:     165353 | Avg value/item: 16535.30
